<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/Diabetes_Prediction_i.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing Libraries**

In [ ]:
# Core dependencies: %pip install numpy pandas matplotlib scikit-learn
# Optional: %pip install imbalanced-learn  (adds a SMOTE comparison)
# Optional: %pip install kagglehub  (downloads the CSV if no local file exists)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, average_precision_score)
RANDOM_STATE = 42

In [ ]:
# Reading the Data: prefer a local diabetes.csv, otherwise download with kagglehub.
DATA_PATH = Path('diabetes.csv')
if not DATA_PATH.exists():
    try:
        import kagglehub
    except ImportError as error:
        raise ImportError('Place diabetes.csv beside this notebook or install kagglehub.') from error
    path = Path(kagglehub.dataset_download('uciml/pima-indians-diabetes-database'))
    DATA_PATH = path / 'diabetes.csv'
# Use an explicit filename rather than whichever file happens to appear first.

In [ ]:
print("Dataset:", DATA_PATH)

# **Data Overview**

In [ ]:
# Creating the DataFrame
feature_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                   'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
df = pd.read_csv(DATA_PATH)
required_columns = feature_columns + ['Outcome']
if not set(required_columns).issubset(df.columns):
    raise ValueError(f'Expected columns: {required_columns}')
df = df[required_columns].copy()
if df['Outcome'].isna().any() or not df['Outcome'].isin([0, 1]).all():
    raise ValueError('Outcome must contain only observed labels 0 and 1.')
if not all(pd.api.types.is_numeric_dtype(df[col]) for col in feature_columns):
    raise ValueError('All model features must be numeric.')
# Exact duplicate records should not appear on both sides of a random split.
print('Exact duplicate records removed:', int(df.duplicated().sum()))
df = df.drop_duplicates().reset_index(drop=True)
df.head(5)

In [ ]:
df.info()

In [ ]:
# Checking Missing Values: isnull() alone does not detect zero-coded missing values.
columns_to_impute = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('Explicit missing values:')
print(df.isnull().sum())
print('Zero-coded values to treat as missing in this dataset:')
print(df[columns_to_impute].eq(0).sum())
# Pregnancy count 0 is valid and must be retained.

In [ ]:
#@title Checking Duplicate Values (Nothing were Found)
df.duplicated().sum()

In [ ]:
df.describe()

### **Observations from Descriptive Statistics (`df.describe()`)**

Here's what we can observe from the mean, standard deviation, min, max, and quartiles for each column:

*   **Pregnancies:**
    *   `Mean`: Approximately 3.8 pregnancies.
    *   `Min`: 0, `Max`: 17. The range is quite wide, indicating a diverse group of individuals in terms of number of pregnancies.
    *   `Quartiles`: 25% of women had 1 or fewer pregnancies, 50% had 3 or fewer, and 75% had 6 or fewer. The distribution seems right-skewed, with fewer women having a very high number of pregnancies.

*   **Glucose, BloodPressure, SkinThickness, Insulin, BMI:**
    *   **Critical Observation:** For all these columns, the `min` value is 0. This is **biologically unrealistic** for `Glucose` (blood sugar), `BloodPressure`, `BMI` (body mass index), and `Insulin` (unless the patient is undergoing specific medical treatment or has a specific condition). `SkinThickness` can legitimately be 0 if the measurement is not taken or is very low, but often 0 here indicates missing data.
    *   These zero values are likely placeholders for missing data and will need to be handled, as they can significantly skew the mean, standard deviation, and any model built on this data. For instance, a `Glucose` level of 0 is impossible for a living person.
    *   `Glucose`: `Mean` 120.9, `Std` 32.0. The wide standard deviation suggests variability. The non-zero minimum for a healthy individual is much higher than 0.
    *   `BloodPressure`: `Mean` 69.1, `Std` 19.4. Similar to Glucose, a blood pressure of 0 is impossible. The mean is within a healthy range, but the standard deviation indicates a good spread.
    *   `SkinThickness`: `Mean` 20.5, `Std` 16.0. Again, a `min` of 0. This could represent missing data, as a person would generally have some skin thickness.
    *   `Insulin`: `Mean` 79.8, `Std` 115.2. The extremely high standard deviation relative to the mean, combined with a `min` of 0 and a `max` of 846, suggests a very skewed distribution, likely with many zero values and some very high outliers. A value of 0 for insulin is also physiologically problematic.
    *   `BMI`: `Mean` 32.0, `Std` 7.9. A `min` of 0 for BMI is impossible. The mean indicates that on average, the population in this dataset is in the obese category (BMI > 30).

*   **DiabetesPedigreeFunction:**
    *   `Mean`: 0.47, `Std`: 0.33. This function assesses the risk of diabetes based on family history. The range from 0.078 to 2.42 suggests varying degrees of genetic predisposition.

*   **Age:**
    *   `Mean`: 33.2, `Std`: 11.8. The `min` is 21 and `max` is 81, showing a broad age range. The quartiles indicate that 75% of individuals are 41 or younger, suggesting a younger-leaning population in this dataset.

*   **Outcome:**
    *   `Mean`: 0.349. This is essentially the proportion of individuals with diabetes (1) in the dataset, which is about 34.9%. This confirms our earlier observation of class imbalance, where roughly one-third of the individuals have diabetes.
    *   `Min`: 0, `Max`: 1, as expected for a binary outcome.

**In Summary:**

The most significant observation is the presence of 0 values in columns like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. These zeros are highly indicative of missing data that have been coded as 0. This is a critical data quality issue that needs to be addressed before any meaningful analysis or model training can be performed. Ignoring these zeros would lead to inaccurate statistics and potentially flawed model predictions.

In [ ]:
# Distribution of Outcome Column (descriptive overview, not a balancing requirement).
outcome_counts = df['Outcome'].value_counts().sort_index()
outcome_percentages = df['Outcome'].value_counts(normalize=True).sort_index() * 100
print('Outcome Distribution (Counts):\n', outcome_counts)
print('Outcome Distribution (Percentages):\n', outcome_percentages.round(2))
outcome_counts.plot.bar(color=['steelblue', 'orange'], rot=0, figsize=(7, 5))
plt.title('Distribution of Diabetes Outcome')
plt.xlabel('Outcome (0 = No Diabetes, 1 = Diabetes)')
plt.ylabel('Number of Cases')
plt.show()

The `Outcome` column represents whether a patient has diabetes (1) or not (0).

From the plot and the counts, we can observe that the dataset is imbalanced. There are significantly more individuals without diabetes (Outcome 0) than with diabetes (Outcome 1). This is a common characteristic in medical datasets and might need to be addressed during model training to prevent the model from being biased towards the majority class.

A `countplot` is generally preferred over a pie chart for showing categorical distributions as it makes it easier to compare the magnitudes of different categories.

# **Data Preprocessing**

In [ ]:
# Splitting into X, Y. Keep the test set separate from all model selection.
x = df[feature_columns].copy()
y = df['Outcome'].astype(int)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Training shapes:', x_train.shape, y_train.shape)
print('Test shapes:', x_test.shape, y_test.shape)
assert x_train.index.equals(y_train.index)
assert x_test.index.equals(y_test.index)
if y_train.value_counts().min() < 5:
    raise ValueError('Need at least five training examples per class for 5-fold CV.')

In [ ]:
# Impute zero-coded missing values inside the model pipeline.
# This rule is specific to these five columns in this dataset, not all zero values.
def replace_missing_zeros(features):
    cleaned = features.copy()
    cleaned[columns_to_impute] = cleaned[columns_to_impute].replace(0, np.nan)
    return cleaned

# The transformer has no learned statistics. The imputer learns training-fold medians.
zero_transformer = FunctionTransformer(replace_missing_zeros, validate=False)
# Do not calculate medians on df: that would include the test rows.

In [ ]:
# Standardization and imputation travel with the model.
# GridSearchCV fits these steps separately inside each training fold.
steps = [
    ('missing_zeros', zero_transformer),
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier()),
]
pipeline = Pipeline(steps)

In [ ]:
# OverSampling (optional): compare against no resampling instead of always using SMOTE.
# imbalanced-learn Pipeline resamples only during fit, never validation/prediction.
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ResamplingPipeline
except ImportError:
    smote_available = False
    print('SMOTE comparison skipped: install imbalanced-learn if you want to include it.')
else:
    smote_available = True
    pipeline = ResamplingPipeline(steps[:-1] + [
        ('sampler', 'passthrough'), steps[-1],
    ])
# Scaling precedes SMOTE because its nearest-neighbor calculations depend on distances.
# Synthetic rows are training aids, not real patients; their probabilities need scrutiny.

In [ ]:
# Verify original training distribution. Validation/test labels are never resampled.
print('Original training class counts:')
print(y_train.value_counts().sort_index())
print('Balancing is a candidate setting; it is not automatically an improvement.')

# **KNN Model**

In [ ]:
# Select settings using 5-fold stratified CV on training data only.
# Macro F1 gives the two classes equal weight. It does not encode clinical costs.
N_neighbors = [1, 3, 5, 7, 9, 11, 13, 15]
parameter_grid = [
    {'model': [DummyClassifier(strategy='most_frequent')]},
    {'model': [KNeighborsClassifier()], 'model__n_neighbors': N_neighbors,
     'model__weights': ['uniform', 'distance']},
    {'model': [LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)],
     'model__C': [0.1, 1.0, 10.0], 'model__class_weight': [None, 'balanced']},
]
if smote_available:
    # Baseline stays unresampled. Compare KNN with/without SMOTE inside every fold.
    # Logistic regression uses class weights as its simpler imbalance comparison.
    for grid in parameter_grid:
        grid['sampler'] = ['passthrough']
    parameter_grid[1]['sampler'] = ['passthrough', SMOTE(random_state=RANDOM_STATE)]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(
    pipeline, parameter_grid, cv=cv,
    scoring={'macro_f1': 'f1_macro', 'accuracy': 'accuracy',
             'recall_positive': 'recall', 'roc_auc': 'roc_auc'},
    refit='macro_f1', n_jobs=1, error_score='raise',
)
search.fit(x_train, y_train)
print('Best settings:', search.best_params_)
print(f'Best mean CV macro F1: {search.best_score_:.4f}')
# This CV score selected the model; it is not an independent final performance estimate.
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[['params', 'mean_test_macro_f1', 'std_test_macro_f1',
                    'mean_test_accuracy', 'mean_test_recall_positive']]
        .sort_values('mean_test_macro_f1', ascending=False).head(10))

In [ ]:
# GridSearchCV already refits the chosen complete pipeline on all training rows.
# Use the actual winner; do not hard-code n_neighbors=1 after searching.
final_model = search.best_estimator_

In [ ]:
# Prediction on test set: run once after all settings are chosen.
y_pred = final_model.predict(x_test)
positive_index = list(final_model.classes_).index(1)
y_score = final_model.predict_proba(x_test)[:, positive_index]
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test ROC AUC: {roc_auc_score(y_test, y_score):.4f}')
print(f'Test average precision: {average_precision_score(y_test, y_score):.4f}')

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])
ClassificationReport = classification_report(
    y_test, y_pred, labels=[0, 1], target_names=['Outcome 0', 'Outcome 1'], zero_division=0)
print('Confusion Matrix:\n', conf_matrix)
print('Classification Report:\n', ClassificationReport)
tn, fp, fn, tp = conf_matrix.ravel()
print('Sensitivity (class 1 recall):', tp / (tp + fn) if tp + fn else float('nan'))
print('Specificity (class 0 recall):', tn / (tn + fp) if tn + fp else float('nan'))
ConfusionMatrixDisplay(conf_matrix, display_labels=['Outcome 0', 'Outcome 1']).plot(cmap='Blues')
plt.show()

## Review notes
- The old imputation cell changed `df` after `x_train`/`x_test` had already been made. The intended replacements did not reach training. Moving this step earlier while still using full-data medians would introduce leakage; the pipeline fixes both problems.
- The original scaler fitted only training data and SMOTE resampled only training rows: those choices were correct. For cross-validation, both preprocessing and any SMOTE must be fitted inside each fold.
- The old neighbor search chose settings using test accuracy, then hard-coded one neighbor. The revised search uses training-only CV and retains the selected complete model.
- Roughly 65/35 class proportions do not automatically require resampling. Compare approaches using class-specific metrics and a majority baseline.
- The retained original descriptive notes below are learning notes, not clinical interpretations. The zero-handling rule applies to this dataset; keep legitimate zero pregnancies and zero Outcome labels.
- The dataset is small. One test split has substantial uncertainty, and the old test set was already used for tuning. A new seed does not create genuinely new patients; external data is needed for a fresh generalization estimate.
- This assumes independent rows. If repeat-patient identifiers become available, split by patient instead. Exact-record deduplication cannot establish patient independence.
- Classifier scores, particularly after resampling, are not automatically calibrated probabilities. Tune thresholds/calibration within training data, never on the final test set.
- Performance on this dataset does not establish diagnostic usefulness or generalization to other populations.

References: [scikit-learn pipeline pitfalls](https://scikit-learn.org/stable/common_pitfalls), [imbalanced-learn resampling pitfalls](https://imbalanced-learn.org/stable/common_pitfalls.html).

## Original comments and learning notes
Changed original cells are preserved below for reference. The executable cells above contain the corrections.

<details>
<summary>Original cell 3</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
```

</details>

<details>
<summary>Original cell 4</summary>

```python
#@title Reading the Data
import kagglehub
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
```

</details>

<details>
<summary>Original cell 5</summary>

```python
file_list = os.listdir(path)
print(file_list)
```

</details>

<details>
<summary>Original cell 7</summary>

```python
#@title Creating the DataFrame
df = pd.read_csv(os.path.join(path, file_list[0]))
df.head(5)
```

</details>

<details>
<summary>Original cell 9</summary>

```python
#@title Checking Missing Values (Nothing were Found)
df.isnull().sum()
```

</details>

<details>
<summary>Original cell 13</summary>

```python
#@title Distribution of 'Outcome' Column
outcome_counts = df['Outcome'].value_counts()
outcome_percentages = df['Outcome'].value_counts(normalize=True) * 100

print("Outcome Distribution (Counts):\n", outcome_counts)
print("\nOutcome Distribution (Percentages):\n", outcome_percentages.round(2))

plt.figure(figsize=(7, 5))
sns.countplot(x='Outcome', data=df, palette='viridis')
plt.title('Distribution of Diabetes Outcome')
plt.xlabel('Outcome (0 = No Diabetes, 1 = Diabetes)')
plt.ylabel('Number of Cases')
plt.show()
```

</details>

<details>
<summary>Original cell 16</summary>

```python
#@title Splitting into X, Y

x = df.drop(['Outcome'], axis=1)
y = df['Outcome']


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

print("Shape of x_train:", x_train.shape)
print("Shape of x_test:", x_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
```

</details>

<details>
<summary>Original cell 17</summary>

```python
#@title Impute Biologically Wrong Values
columns_to_impute = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for column in columns_to_impute:
    # Calculate the median of the column, excluding zeros
    median_val = df[df[column] != 0][column].median()
    # Replace zeros with the calculated median
    df[column] = df[column].replace(0, median_val)

print("DataFrame after imputing zero values:")
print(df.head())
print("\nDescriptive statistics after imputation:")
print(df.describe())
```

</details>

<details>
<summary>Original cell 18</summary>

```python
#@title Standardization
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
```

</details>

<details>
<summary>Original cell 19</summary>

```python
#@title OverSampling
smote = SMOTE(random_state=0)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

print("Shape of x_train_resampled:", x_train_resampled.shape)
print("Shape of y_train_resampled:", y_train_resampled.shape)
print("Class distribution of y_train_resampled:\n", pd.Series(y_train_resampled).value_counts())
```

</details>

<details>
<summary>Original cell 20</summary>

```python
#@title Verify Balanced Data Distribution
print("Class distribution of y_train_resampled:")
print(pd.Series(y_train_resampled).value_counts())

plt.figure(figsize=(7, 5))
sns.countplot(x=pd.Series(y_train_resampled), palette='viridis')
plt.title('Distribution of Outcome in Oversampled Training Data')
plt.xlabel('Outcome (0 = No Diabetes, 1 = Diabetes)')
plt.ylabel('Number of Cases')
plt.show()
```

</details>

<details>
<summary>Original cell 22</summary>

```python
N_neighbors = [1, 3, 5, 7, 9, 11, 13, 15]

best_accuracy = 0
best_N = None

for i in N_neighbors:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(x_train_resampled, y_train_resampled)
    y_pred_current = knn.predict(x_test)
    current_accuracy = accuracy_score(y_test, y_pred_current)

    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        best_N = i

print(f"Best N_neighbors: {best_N}")
print(f"Best Accuracy: {best_accuracy:.4f}")
```

</details>

<details>
<summary>Original cell 23</summary>

```python
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(x_train_resampled, y_train_resampled)
```

</details>

<details>
<summary>Original cell 24</summary>

```python
y_pred = knn.predict(x_test) # Prediction on test set

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
```

</details>

<details>
<summary>Original cell 25</summary>

```python
conf_matrix = confusion_matrix(y_test, y_pred)
ClassificationReport = classification_report(y_test, y_pred)

print("Confusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", ClassificationReport)
```

</details>